# 09 — Descriptive Statistics for Data Section

Produces `descriptive_stats.json` and `descriptive_stats.csv` with all summary statistics
needed for the paper's data table. Uses the same warmup as the main panel.

In [ ]:
import sys; sys.path.insert(0, "..")
from config import CFG, ECON_DIR
import pandas as pd
import numpy as np
import json

# ── Load panel and apply NB07 warmup ──
df = pd.read_parquet(CFG.FILES.econ_core_full, engine="pyarrow")
df["date"] = pd.to_datetime(df["date"], utc=True)
df = df.sort_values("date").reset_index(drop=True)

warmup = max(CFG.ECON.vol_window, 720) + max(CFG.ECON.lp_horizons) + 2
df_est = df.iloc[warmup:].copy()

# ── 1. Standard descriptive stats ──
variables = {
    "ret_eth_perp": "ret_eth_perp",
    "ret_btc_spot": "ret_btc_spot",
    "vol_eth_7d": "vol_eth_7d",
    "funding_rate": "funding_rate",
    "basis_bps": "basis_bps",
    "oi": "oi",
    "log_liq": "log_liq",
}

stats_json = {}
rows_csv = []

for label, col in variables.items():
    s = df_est[col]
    d = {
        "N": int(s.notna().sum()),
        "mean": float(s.mean()),
        "std": float(s.std()),
        "min": float(s.min()),
        "p1": float(s.quantile(0.01)),
        "p5": float(s.quantile(0.05)),
        "p25": float(s.quantile(0.25)),
        "p50": float(s.quantile(0.50)),
        "p75": float(s.quantile(0.75)),
        "p95": float(s.quantile(0.95)),
        "p99": float(s.quantile(0.99)),
        "max": float(s.max()),
    }
    if label == "log_liq":
        nonzero = (df_est["liq_usd_total"] > 0).sum()
        d["n_nonzero"] = int(nonzero)
        d["pct_nonzero"] = round(float(nonzero / len(df_est) * 100), 2)
    stats_json[label] = d
    for stat, val in d.items():
        rows_csv.append({"variable": label, "statistic": stat, "value": val})

# ── 2. liq_usd_total (raw USD) ──
liq_raw = df_est["liq_usd_total"]
liq_d = {
    "sum_total_usd": float(liq_raw.sum()),
    "mean_nonzero_usd": float(liq_raw[liq_raw > 0].mean()),
}
df_est_tmp = df_est.copy()
df_est_tmp["year"] = df_est_tmp["date"].dt.year
for yr, grp in df_est_tmp.groupby("year"):
    liq_d[f"sum_{yr}_usd"] = float(grp["liq_usd_total"].sum())

stats_json["liq_usd_total"] = liq_d
for stat, val in liq_d.items():
    rows_csv.append({"variable": "liq_usd_total", "statistic": stat, "value": val})

# ── 3. Period and sample size ──
period_d = {
    "first_obs": str(df["date"].iloc[0]),
    "last_obs": str(df["date"].iloc[-1]),
    "N_total": int(len(df)),
    "N_after_warmup": int(len(df_est)),
    "warmup_rows": int(warmup),
}
stats_json["sample"] = period_d
for stat, val in period_d.items():
    rows_csv.append({"variable": "sample", "statistic": stat, "value": val})

# ── 4. oi_high indicator ──
oi_high_pct = float(df_est["oi_high"].mean() * 100)
stats_json["oi_high"] = {"pct_high": round(oi_high_pct, 2)}
rows_csv.append({"variable": "oi_high", "statistic": "pct_high", "value": round(oi_high_pct, 2)})

# ── 5. Anchor checks ──
# Order-of-magnitude cross-checks of two headline values reported in the
# paper's data section: std(ret_eth_perp) ~ 0.8 %/h and cumulative
# liquidations ~ 2.5 bn USD.
std_ret = stats_json["ret_eth_perp"]["std"]
sum_liq = stats_json["liq_usd_total"]["sum_total_usd"]

anchors = {}
anchors["std_ret_eth_perp"] = {
    "value": round(std_ret, 4),
    "anchor_estimate": 0.8,
    "deviation_pct": round(abs(std_ret - 0.8) / 0.8 * 100, 1),
    "anchor_discrepancy": abs(std_ret - 0.8) / 0.8 > 0.20,
}
sum_liq_bn = sum_liq / 1e9
anchors["cumulative_liq_usd"] = {
    "value_bn": round(sum_liq_bn, 3),
    "anchor_estimate_bn": 2.5,
    "deviation_pct": round(abs(sum_liq_bn - 2.5) / 2.5 * 100, 1),
    "anchor_discrepancy": abs(sum_liq_bn - 2.5) / 2.5 > 0.20,
}
stats_json["anchor_checks"] = anchors

for check_name, check_d in anchors.items():
    for stat, val in check_d.items():
        rows_csv.append({"variable": f"anchor_{check_name}", "statistic": stat, "value": val})

# ── Save ──
with open(ECON_DIR / "descriptive_stats.json", "w") as f:
    json.dump(stats_json, f, indent=2)

df_csv = pd.DataFrame(rows_csv)
df_csv.to_csv(ECON_DIR / "descriptive_stats.csv", index=False)

# ── Display ──
print("=== Anchor checks ===")
for k, v in anchors.items():
    flag = " *** DISCREPANCY" if v.get("anchor_discrepancy") else ""
    print(f"  {k}: {v}{flag}")

print(f"\n=== CSV shape: {df_csv.shape} ===")
print(df_csv.to_string(index=False))
